<a href="https://colab.research.google.com/github/KarimBekhtiGalvao/Diffusion-Based-Generation/blob/main/baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yaml
from google.colab import drive
import os
import re
import unicodedata
from collections import Counter
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import TensorDataset, DataLoader
import math
from IPython.utils.py3compat import no_code
from torch.nn import TransformerEncoder, TransformerEncoderLayer

drive.mount('/content/drive')
data_dir = "/content/drive/MyDrive/Cours/CAI_projet"
cwd = os.chdir(data_dir)
params = yaml.safe_load(open("params.yaml", "r", encoding="utf-8"))
dtype_map = {
    "torch.long": torch.long,
    "torch.float32": torch.float32,
    "torch.float": torch.float,
    "torch.int64": torch.int64,
}
params["dtype"] = dtype_map[params["dtype"]]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Introduction

The purpose of this file is to implement a baseline model to compare to our diffusion model. For that , we take an autoregressive transformer, trained with the same setups as the diffusion model.


# Load the vocab
We prepared the text input and the vocabulray in Data_preparation.ipynb. We reuse the same code and load the vacabulary here

In [ ]:
### [EXTRACT OF CODE ADAPTED FROM LAB 5]

class CharBPETokenizer:
    def __init__(self, vocab_size=100):

        # Maximum vocabulary size for BPE
        self.vocab_size = vocab_size

        # Set to store current vocabulary
        self.vocab = set()

        # Special tokens used by BERT
        self.special_tokens = ["[UNK]", "[MASK]", "[SEP]"]

    def train(self, text):
        """
        Learn Byte-Pair Encoding (BPE) merges from the input text.
        BPE merges frequent consecutive character pairs into new tokens
        until the vocabulary reaches the desired size.
        """

        # Convert text to list of characters
        tokens = [c for word in text for c in word] # Your code here. Aim for 1 line

        # Initialize the vocabulary with the characters
        vocab = Counter(tokens)

        # Iteratively merge frequent pairs until vocab_size is reached
        while len(vocab) < self.vocab_size:

            pair_counts = Counter()

            # Count how many times each consecutive pair of tokens occurs in the text
            # The result, pair_counts, is a dictionary mapping each pair to its frequency
            # Your code here. Aim for 3-4 lines. 1
            for i in range(len(tokens)-1):
              pair = (tokens[i], tokens[i+1])
              pair_counts[pair] += 1

            if not pair_counts:
                break  # No more pairs to merge

            # Find the most frequent pair
            #print(pair_counts.most_common()) # [(('a', 'a'), 4), (('a', 'b'), 2), (('b', 'd'), 1), (('d', 'a'), 1), (('b', 'a'), 1), (('a', 'c'), 1)]
            best_pair, freq = pair_counts.most_common(1)[0]# Your code here. Use the .most_common method.

            # Merge the most frequent pair into a new token
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == best_pair:
                    new_tokens.append(tokens[i] + tokens[i + 1]) # Append new token # Your code here. Aim for 1 line
                    i += 2# Skip merged token  # Your code here. Aim for 1 line
                else:
                    new_tokens.append(tokens[i]) # Append old token
                    i += 1# Go to the next token # Your code here. Aim for 1 line

            tokens = new_tokens
            vocab = Counter(tokens)


        # Combine learned tokens, original characters, and special tokens
        vocab_set = set(tokens) | set(text) | set(self.special_tokens)

        # Sort vocab_set by length (longest first)
        self.vocab_list = sorted(vocab_set, key=len, reverse=True) # Your code here. Aim for 1 line.

        # Mapping from token to ID with a vocabulary
        self.token_to_id = {token: i for i, token in enumerate(self.vocab_list)}# Your code here. Aim for a line.

        # Mapping from Id to tokens with a vocabulary
        self.id_to_token = {i: token for i, token in enumerate(self.vocab_list)}# Your code here. Aim for a line.


    def tokenize(self, text):
        """
        Tokenize input text. We replace longest tokens first.
        If a character sequence is not in the vocabulary, replace it with [UNK].
        """
        tokens = []
        i = 0

        while i < len(text):
            match_found = False

            # Try to match the longest token first
            for token in self.vocab_list:
                if text.startswith(token, i):
                    tokens.append(token)
                    i += len(token)# Go to the next token. Your code here. Aim for 1 line.
                    match_found = True
                    break
            if not match_found:
                tokens.append("[UNK]")  # Replace unknown character
                print(f"[WARNING] Character '{text[i]}' not in vocabulary, replaced with [UNK]")
                i += 1# Go to the next token. Your code here. Aim for 1 line.

        return tokens

    def encode_ids(self, text):
        """
        Convert a text string into a list of token IDs
        """
        # Convert raw text into tokens
        tokens = self.tokenize(text)# Your code here. Aim for 1 line.

        # Assing index to each token
        token_ids = [self.token_to_id[t] for t in tokens]# Your code here. Aim for 1 line.
        return token_ids

    def detokenize(self, tokens):
        """
        Convert a list of tokens back into a text string
        """
        return "".join(tokens)

    def decode_ids(self, ids):
        """
        Convert a list of token IDs back into text
        """
        # Convert ids to tokens
        tokens = [self.id_to_token[i] for i in ids] # Your code here. Aim for 1 line

        # Conver tokens into text (detokenize)
        text =  "".join(tokens) # Your code here. Aim for 1 line
        return text

    def load_vocab_from_txt(self, filename):
        """
        @author Karim Bekhti
        Load a vocabulary from a text file and rebuild tokenizer mappings.
        Assumes one token per line.
        """
        with open(filename, "r", encoding="utf-8") as f:
            self.vocab_list = [line.rstrip("\n") for line in f if line.rstrip("\n") != ""]

        self.vocab = set(self.vocab_list)
        self.token_to_id = {token: i for i, token in enumerate(self.vocab_list)}
        self.id_to_token = {i: token for i, token in enumerate(self.vocab_list)}
        self.vocab_size = len(self.vocab_list)

In [ ]:
vocab_size = params["vocab_size"]
tokenizer = CharBPETokenizer()
tokenizer.load_vocab_from_txt(f"vocab_{vocab_size}.txt")

In [ ]:
tokenizer.tokenize("Emma Woodhouse, handsome, clever, and rich, with a comfortable home and happy disposition, seemed to unite some of the best blessings")

[WARNING] Character 'E' not in vocabulary, replaced with [UNK]
[WARNING] Character 'W' not in vocabulary, replaced with [UNK]


['[UNK]',
 'm',
 'm',
 'a',
 ' ',
 '[UNK]',
 'o',
 'o',
 'd',
 'h',
 'ou',
 's',
 'e',
 ', ',
 'ha',
 'n',
 'd',
 's',
 'om',
 'e',
 ', ',
 'c',
 'le',
 'ver',
 ', ',
 'an',
 'd ',
 'r',
 'i',
 'ch',
 ', ',
 'wi',
 'th',
 ' a ',
 'c',
 'om',
 'for',
 't',
 'a',
 'b',
 'le',
 ' ',
 'h',
 'om',
 'e ',
 'an',
 'd ',
 'ha',
 'p',
 'p',
 'y ',
 'd',
 'is',
 'p',
 'o',
 's',
 'it',
 'ion',
 ', ',
 's',
 'e',
 'e',
 'm',
 'ed ',
 't',
 'o ',
 'u',
 'n',
 'it',
 'e ',
 's',
 'om',
 'e ',
 'of ',
 'th',
 'e ',
 'be',
 'st',
 ' ',
 'b',
 'le',
 's',
 's',
 'ing',
 's']

Once loaded, we can take a sequence and transform it into its constitutive tokens. As in lab 6, we prepare the data by chunking it into bits of 100 characters. We then define the sets of train, test and split, each with proportion determined in the params.yaml

In [ ]:
# Read the text file
with open("texts_normalized.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Split into chunks of 100 characters
chunks = [text[i:i+100] for i in range(0, len(text), 100)]

print("Number of chunks:", len(chunks))
print("First chunk:", chunks[0])
stop = 1261
length = len(chunks[:stop])
encoded_chunks = [tokenizer.encode_ids(chunk) for chunk in chunks[:stop]]

Number of chunks: 126142
First chunk: volume i chapter i emma woodhouse, handsome, clever, and rich, with a comfortable home and happy dis


Now the dataset is completely encoded. With its length we calculate the index of end of each subset (train for the training, test for testing)



In [ ]:
train_end = int(length * params["train_size"])
test_end = train_end + int(length*params["test_size"])

train_data = encoded_chunks[:train_end]
test_data = encoded_chunks[train_end:test_end]

train_data = [torch.tensor(seq, dtype=params["dtype"]) for seq in train_data]
test_data = [torch.tensor(seq, dtype=params["dtype"]) for seq in test_data]


train_tensors = [torch.tensor(seq, dtype=params["dtype"]) for seq in train_data]
test_tensors = [torch.tensor(seq, dtype=params["dtype"]) for seq in test_data]

train_padded = pad_sequence(train_tensors, batch_first=True, padding_value=0)
test_padded = pad_sequence(test_tensors, batch_first=True, padding_value=0)

tr_dataset = TensorDataset(train_padded)
te_dataset = TensorDataset(test_padded)

tr_loader = DataLoader(tr_dataset, batch_size=params["batch_size"], shuffle=True)
te_loader = DataLoader(te_dataset, batch_size=params["batch_size"], shuffle=False)


/tmp/ipykernel_21199/1955782353.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_tensors = [torch.tensor(seq, dtype=params["dtype"]) for seq in train_data]
/tmp/ipykernel_21199/1955782353.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_tensors = [torch.tensor(seq, dtype=params["dtype"]) for seq in test_data]


# Embeddings

We create Positional Embeddings for our transformer. For the embeddings, we use the torch.nn.Embedding function native to torch to save some time.

We use the torch.nn.CrossEntropyLoss, Adam optimizer.

In [ ]:
class PositionalEncoding(torch.nn.Module):

    def __init__(self, d_model, max_len=500):
        """
        Inputs
            d_model - Hidden dimensionality of the input.
            max_len - Maximum length of a sequence to expect.
        """
        super().__init__()

        # Create matrix of [SeqLen, HiddenDim] representing the positional encoding for max_len inputs
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        # register_buffer => Tensor which is not a parameter, but should be part of the modules state.
        # Used for tensors that need to be on the same device as the module.
        # persistent=False tells PyTorch to not add the buffer to the state dict (e.g. when we save the model)
        self.register_buffer('pe', pe, persistent=False)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

To keep the decoder from attending to future elements, we build this masking function

In [ ]:
def generate_mask(L):
    """Generates an upper-triangular matrix of -inf, with zeros on diag."""
    return  torch.triu(torch.full((L, L), float("-inf")), diagonal=1)

In [ ]:


class TransformerLM(torch.nn.Module):

    def __init__(self, ntoken, emsize, nhead, d_model, nlayers, dropout):
        super().__init__()
        self.pos_encoder = PositionalEncoding(d_model=d_model)# Your code here. Aim for 1 line.
        self.encoder = torch.nn.Embedding(ntoken, emsize)    # Your code here. Aim for 1 line.
        encoder_layers = TransformerEncoderLayer(d_model=d_model, nhead = nhead, dim_feedforward=d_model, batch_first=True, dropout=dropout) # Your code here. Aim for 1 line.
        self.transformer_encoder = TransformerEncoder(encoder_layers, nlayers) # Your code here. Aim for 1 line.
        self.linear = torch.nn.Linear(d_model, ntoken)  # Your code here. Aim for 1 line.

    def forward(self, src, src_mask):
        """
        Args:
            src: Tensor, shape [batch_size, seq_len]
            src_mask: Tensor, shape [seq_len, seq_len]

        Returns:
            output Tensor of shape [batch_size, seq_len, ntoken]
        """

        # Your code here. Aim for 4 lines
        out = self.encoder(src)
        out = self.pos_encoder(out)
        out = self.transformer_encoder(out, mask = src_mask)
        output = self.linear(out)

        return output

In [ ]:
emsize = 200  # embedding dimension
d_model = 200  # dimension of the model
nlayers = 4  # number of nn.TransformerEncoderLayer in nn.TransformerEncoder
nhead = 8  # number of heads in nn.MultiheadAttention
dropout = 0.2  # dropout probability
batch_size = params["batch_size"]
vocab_size = tokenizer.vocab_size

model = TransformerLM(vocab_size, emsize, nhead, d_model, nlayers, dropout)

In [ ]:
# Hyperparameters
num_epoch = 50
lr = 0.001
device = 'cuda' # use 'cpu' for debugging (change the runtime type accordingly)
#model.to(device)

loss = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
out_len = 100
src_mask = generate_mask(out_len)#.to(device)

def training_loop(X, model, loss, optimizer, num_epoch=500, batch_size=100, device='cpu'):

  # Training Loop
  for epoch in range(num_epoch):
    print(epoch)
    for i, X in enumerate(tr_loader):

      # Input/Labels Manipulation
      X = X[0] # X is always a list composed of 1 tensor #torch.Size([128, 101]) # B L
      #print(X.shape)
      y = X[:,1:]# Your code here. Aim for 1 line.
      #print(y.shape) #torch.Size([128, 100]) B x L - 1
      X=X[:,:-1]
      #y=y[:,:-1]
      #print(X.shape)
      X = X#.to(device)
      y = y#.to(device)

      # Run the Model
      out = model(X, generate_mask(X.shape[1]))# Your code here. Aim for 1 line


      # Compute the loss
      #print(out.shape) #torch.Size([128, 100, 66])
      #print(y.shape) #torch.Size([128, 100])
      l = loss(out.transpose(1, 2), y)# Your code here. Aim for 1 line

      # Update the parameters
      # Your code here. Aim for 3 lines
      optimizer.zero_grad()
      l.backward()
      optimizer.step()

    print(l.item())
    # Print loss
    if (epoch + 1) % 10 == 0:
        print("Epoch %03d: Train_loss: %.4f " %(epoch+1, l.item()))

# Run the training loop
training_loop(train_data , model, loss, optimizer, num_epoch, batch_size, device)

0
3.693699359893799
1
3.521542549133301
2
3.3763020038604736
3
3.1755316257476807
4
3.0193376541137695
5
2.928954601287842
6
2.825333595275879
7
2.7536048889160156
8
2.743058919906616
9
2.6896514892578125
Epoch 010: Train_loss: 2.6897 
10
2.640432357788086
11
2.604722499847412
12
2.577517032623291
13
2.5246381759643555
14
2.5238823890686035
15
2.4591169357299805
16
2.4277634620666504
17
2.3758771419525146
18
2.3425168991088867
19
2.3023908138275146
Epoch 020: Train_loss: 2.3024 
20
2.2741899490356445
21
2.2608141899108887
22
2.217724323272705
23
2.2350211143493652
24
2.1174111366271973
25
2.0990922451019287
26
2.0792670249938965
27
2.0481531620025635
28
2.014740467071533
29
1.9796863794326782
Epoch 030: Train_loss: 1.9797 
30
1.9052470922470093
31
1.9282439947128296
32
1.873124361038208
33
1.861042857170105
34
1.851064920425415
35
1.7852818965911865
36
1.7855788469314575
37
1.7743321657180786
38
1.7514971494674683
39
1.6894476413726807
Epoch 040: Train_loss: 1.6894 
40
1.71511065959930

In [ ]:
# Switch model to eval modality
# Your code here. Aim for 1 line
model.eval()

# Put test data on the right device
test_data = test_data#.to(device)# Your code here. Aim for 1 line

# Select 128 sentences
X_test = test_data[0:128] # Select 128 sentences

# Derive labels (right shift)
x_list = [seq[:-1].long() for seq in X_test if len(seq) > 1]
y_list = [seq[1:].long() for seq in X_test if len(seq) > 1]

X_test = pad_sequence(x_list, batch_first=True, padding_value=0)#.to(device)
y_test = pad_sequence(y_list, batch_first=True, padding_value=0)#.to(device)

#y_test=y_test[:,:-1]

# Running the model on test data
with torch.no_grad():
  out = model(X_test, generate_mask(X_test.shape[1]))#.to(device))# Your code here. Aim for 1 line

  # Compute the loss
  loss_test = loss(out.transpose(1, 2), y_test) # Your code here. Aim for 1 line

# Compute the loss
#loss_test = # Your code here. Aim for 1 line

# Print the test loss
print("Test_loss: %.4f " %(loss_test.item()))

Test_loss: 2.5610 


In [ ]:
def generate(model, tokenizer, L=100, temp=1.0, device="cpu"):
    model.eval()

    # start token id
    current_input = [0]
    output_text = ""

    with torch.no_grad():
        for i in range(L):
            src_mask = generate_mask(len(current_input))#.to(device)

            inp_tensor = torch.tensor(current_input, dtype=torch.long).unsqueeze(0) #device=device).unsqueeze(0)
            out = model(inp_tensor, src_mask)

            # logits for last position
            logits = out[:, -1, :] / temp

            # block special tokens if desired
            for tok in ["[MASK]", "[SEP]", "[UNK]"]:
                if tok in tokenizer.token_to_id:
                    logits[0, tokenizer.token_to_id[tok]] = -float("inf")

            probs = torch.softmax(logits, dim=-1)
            current_output = torch.multinomial(probs, num_samples=1).item()

            output_text += tokenizer.id_to_token[current_output]
            current_input.append(current_output)

    return output_text
text = generate(model, tokenizer, L=100, temp=1, device=device)
print(text)

eans were a wards, awhere camiat venting of a proy eto beart it isit." "i have pretuch ondone to nojhould shorto harequt of.-tum," have ! pl itshave been
